# HURDLER task query

Runs one repeat module across the eight frozen plasmids using the same artifact as validation and success-rate analyses.

**Rules:** `legacy-optimized-v1`; **seed:** 42 unless explicitly noted.

In [ ]:
REPO = '/home/wendai/projects/hurdler/clone_repeat_protein'
RULE_PROFILE = 'legacy-optimized-v1'
INDEX_DIR = '/net/scratch/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step01_reference_lookup/runs/run01_production/raw/legacy-optimized-v1'
MODULE = 'VLA'

In [ ]:
from pathlib import Path
import hashlib, json
import pandas as pd

def sha256(path):
    path = Path(path)
    if not path.is_file(): return None
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''): h.update(chunk)
    return h.hexdigest()

run_context = {'rule_profile': RULE_PROFILE, 'input_hashes': {}, 'row_counts': {}, 'filter_flow': [], 'limitations': []}

In [ ]:
from hurdler.index import PatternIndex
from hurdler.matching import materialize_best_solution, query_all_plasmids
index = PatternIndex.load(INDEX_DIR)
rows = [materialize_best_solution(result, index) for result in query_all_plasmids(MODULE, index)]
run_context['input_hashes']['pattern_index.npz'] = sha256(Path(INDEX_DIR) / 'pattern_index.npz')
results = pd.DataFrame(rows)
run_context['row_counts'] = {'plasmids_tested': len(results), 'successful_plasmids': int(results.success.sum())}
run_context['filter_flow'] = ['expand motifs shorter than 6AA', 'scan doubled module', 'return first frozen-order match per plasmid']
results

In [ ]:
results.groupby('success').size().rename('plasmids').to_frame()